In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from datascience_toolkit import (
    EDAQuick,
    TimeSeriesDiagnostics,
    TimeSeriesFeatureExtractor,
    TimeSeriesForecaster,
)

In [7]:
df_agua = pd.read_csv("../data/consumo_agua.csv", sep=";")
print(df_agua.shape)
df_agua.head(3)

(1005, 2)


,fecha,valor
0,2019-03-02,1771.416
1,2019-03-03,3163.136
2,2019-03-04,3139.684


In [9]:
TimeSeriesDiagnostics.stationarity_test(df_agua["valor"])

C:\Users\VíctorAguirreVargas\OneDrive - Kepler Group LLC\Documents\DS&A\Data Science\Paycom Tests\Datascience-Stuff\datascience_toolkit\timeseries\diagnostics.py:60: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_res = kpss(clean_s, regression="c", nlags="auto")


,test,statistic,p_value,critical_5pct,is_stationary,interpretation
0,Augmented Dickey-Fuller (ADF),-2.9988,0.035,-2.8645,True,Stationary (Reject H0)
1,KPSS Test,5.6296,0.010,0.4630,False,Non-Stationary (Reject H0)


In [14]:
# 1. Asegurar frecuencia diaria ("D") y split de 30 días para test
ts = TimeSeriesForecaster(
    df=df_agua,
    date_col="fecha",      # Mantén el nombre exacto de tu columna de fecha
    target_col="valor",  # Mantén el nombre exacto de tu columna de consumo
    freq="D"
)

ts.to_series(agg="mean")
ts.prepare(test_size=30)

In [15]:
# 2. Verificar periodos con el periodograma (debería detectar 7 días)
periodos = ts.periodogram_candidates(best=3, min_period=2, max_period=30)
print("Periodos detectados (días):", periodos)

# 3. Registrar modelos y calibrar con estacionalidad semanal (S = 7)
ts.add_default_models(
    include_lstm=False,
    include_cnn=False,
    include_prophet=True,
    include_xgb=True
)

Periodos detectados (días): [7, 4, 2]


C:\Users\VíctorAguirreVargas\OneDrive - Kepler Group LLC\Documents\DS&A\Data Science\Paycom Tests\Datascience-Stuff\datascience_toolkit\timeseries\forecasting.py:328: UserWarning: Prophet no instalado; se omite PROPHET.
  warnings.warn("Prophet no instalado; se omite PROPHET.")


In [16]:
# Holt-Winters aditivo con ciclo de 7 días
ts.tune("ETS", trend="add", seasonal="add", seasonal_periods_candidates=[7], paso=0.2)

# SARIMA estacional con S = 7 (búsqueda aleatoria rápida de 25 combinaciones)
ts.tune("SARIMA", S_candidates=[7], search="random", max_evals=25, max_p=2, max_d=1, max_q=2)

# XGBoost con rezagos semanales (7, 14, 21 días)
if "XGBTS" in ts.models_:
    ts.tune(
        "XGBTS",
        search="random",
        max_evals=25,
        lag_sets=[[1, 2, 3, 7, 14], [1, 7, 14, 21]],
        roll_sets=[[7, 14], [7, 14, 28]]
    )

if "PROPHET" in ts.models_:
    ts.tune("PROPHET", search="random", max_evals=15, yearly_seasonality="auto", weekly_seasonality=True, daily_seasonality=False)

# 4. Competir, reentrenar con toda la serie y graficar
lb = ts.compete(metric="rmse", verbose=True)
ts.fit(model="best", refit_on="train_test_full")
ts.plot_forecast(steps=30, show_test_pred=True)

Leaderboard (RMSE) — menor es mejor


,model,rmse,params_str
0,SARIMA,443.7875,"{""order"": [2, 1, 2], ""seasonal_order"": [0, 1, 0, 7]}"
1,ETS,486.3910,"{""alpha"": 0.2, ""beta"": 0.2, ""gamma"": 0.6, ""optimized"": false, ""seasonal"": ""add"", ""seasonal_periods"": 7, ""trend"": ""add""}"
2,XGBTS,959.7077,"{""colsample_bytree"": 0.9, ""lags"": [1, 7, 14, 21], ""learning_rate"": 0.05, ""max_depth"": 6, ""n_estimators"": 300, ""reg_alpha"": 0.0, ""reg_lambda"": 1.0, ""rolling_windows"": [7, 14], ""subsample"": 1.0, ""use_date_parts"": true}"


Best model params


,model,order,seasonal_order
0,SARIMA,"(2, 1, 2)","(0, 1, 0, 7)"


Target distribution (ALL)


,min,q1,median,q3,max,iqr
0,684.3960,3184.8280,3782.2600,4240.6080,4808.0320,1055.7800


Target distribution (TEST)


,min,q1,median,q3,max,iqr
0,1727.7240,4205.6610,4509.3160,4573.2370,4678.7600,367.5760


RMSE contextualizado (ALL)


,rmse,rmse_over_iqr,rmse_over_range
0,443.7875,0.4203,0.1076
